# 유음화/비음화 환경 검색 (ㄴㄹ, ㄹㄴ, 비음+ㄹ, 저해음+ㄹ)

**작성일**: 2026-02-10  
**목적**: 유음화/비음화 관련 음운 환경 검색  
**관련**: 34_n_insertion_v2.ipynb

---

## 🎯 검색 대상

### 1. 형태소 경계 (dict_morph 기준)

| 환경 | 로마자 | 예시 | 현상 |
|------|--------|------|------|
| ㄴ+ㄹ | n + l/r | 신-라 | 유음화 [실라] |
| ㄹ+ㄴ | l + n | 칼-날 | 유음화 [칼랄] |
| 비음+ㄹ | m/n/ng + l/r | 심-리, 진-리, 강-력 | 유음화 |
| 저해음+ㄹ | k/t/p/s/ch + l/r | 독-립, 출-력 | 비음화 또는 유지 |

### 2. 단어 내부 (형태소 경계 없음)

- 같은 환경이지만 형태소 경계가 없는 경우
- 예: 관-리 (형태소 경계) vs 날-리다의 '날리' (내부)

---

## 📊 출력 정보

### 기본 정보
- word, sense_id, definition

### 환경 정보
- env_type: ㄴㄹ / ㄹㄴ / 비음ㄹ / 저해음ㄹ
- boundary_type: morpheme / internal
- consonant1, consonant2 (로마자)

### 형태소/음절 정보
- dict_morph, word_roman
- seg_morph, anal_morph (비교용)

### 발음 정보
- pron, pron_roman

### 빈도 정보
- freq_LS_total, freq_MP_total (NIKL 코퍼스)
- freq_06b, freq_13a (강범모·김흥규 2009)

---

## 1️⃣ 환경 설정

In [1]:
# 1.1 Google Drive 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("로컬 환경")

Mounted at /content/drive


In [2]:
# 1.2 경로 설정
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
else:
    PROJECT_ROOT = 'g:/내 드라이브/DATA_2026'

V7_LEXICON = f'{PROJECT_ROOT}/10_dictionary_build/output/04_v7_lexicon.csv'
RESULT_DIR = f'{PROJECT_ROOT}/30_search_dictionary/search_results'

print(f"v7: {V7_LEXICON}")
print(f"결과: {RESULT_DIR}")

v7: /content/drive/MyDrive/DATA_2026/10_dictionary_build/output/04_v7_lexicon.csv
결과: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results


In [3]:
# 1.3 라이브러리
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import re
import os

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

# 공통 유틸리티 로드
os.chdir(f'{PROJECT_ROOT}/30_search_dictionary')
%run utils_phonology.py

[OK] utils_phonology.py 로드 완료
   함수 34개


---

## 2️⃣ 데이터 로드

In [4]:
# 2.1 v7 Lexicon 로드
df_v7 = pd.read_csv(V7_LEXICON, encoding='utf-8-sig', low_memory=False)
print(f"v7 Lexicon: {len(df_v7):,}개")
print(f"dict_morph 있는 행: {df_v7['dict_morph'].notna().sum():,}개")

v7 Lexicon: 528,088개
dict_morph 있는 행: 323,146개


# 2.2 형태소 어종 딕셔너리 구축 ⭐

**중요**: 각 형태소의 어종을 정확하게 파악하기 위해 v7에서 단일 형태소 추출

In [5]:
# 어종 딕셔너리 구축 (sense_no 기반)
sense_origin_dict = build_sense_origin_dict(df_v7)

sense_no 어종 딕셔너리: 74개


---

## 3️⃣ 로마자 기반 분석 함수

In [6]:
# 3.1 자음 분류 함수 — utils_phonology.py에서 로드됨
# get_consonant_type, ends_with_consonant_type, starts_with_consonant_type

In [7]:
# 3.2 형태소 파싱 — utils_phonology.py에서 로드됨
# parse_dict_morph, parse_dict_morph_with_boundaries, parse_word_roman_by_morphemes

In [8]:
# 3.3 합성어 유형 분류 — utils_phonology.py에서 로드됨
# classify_compound_type, get_compound_type_description, classify_compound_type_from_row

---

## 4️⃣ 환경 검색 함수

In [9]:
# 4.1 형태소 경계에서 환경 확인 — utils_phonology.py에서 로드됨
# check_nl_ln_at_boundary (장애음ㄴ, 장애음ㅁ 환경 포함)

In [10]:
# 4.2 단어 내부에서 환경 확인 — utils_phonology.py에서 로드됨
# check_nl_ln_internal (장애음ㄴ, 장애음ㅁ 환경 포함)

---

## 5️⃣ 검색 실행

In [11]:
# 5.1 검색 함수
def search_nl_ln_candidates(df, sense_origin_dict):
    """
    ㄴㄹ, ㄹㄴ, 비음+ㄹ, 저해음+ㄹ, 장애음+ㄴ, 장애음+ㅁ 환경 검색

    형태소 경계와 단어 내부를 모두 검색
    - sense_origin_dict: sense_no 기반 어종 딕셔너리 (v7에서 구축)
    - classify_compound_type_from_row: word_type + seg_links 기반 어종 분류
    - expected_process / detected_process: 이론적 예상 vs 실제 발음 감지
    """
    candidates = []

    # 명사만 필터링
    df_noun = df[df['pos'] == '명사'].copy()
    print(f"명사: {len(df_noun):,}개")

    # word_roman이 있는 것만
    df_noun = df_noun[df_noun['word_roman'].notna()]
    print(f"word_roman 있는 명사: {len(df_noun):,}개")

    for idx, row in df_noun.iterrows():
        word = row['word']
        dict_morph = row.get('dict_morph', '')
        word_roman = row['word_roman']
        pron_roman = str(row.get('pron_roman', '')) if pd.notna(row.get('pron_roman', '')) else ''

        # 1. 형태소 경계 검색 (dict_morph가 있는 경우)
        if pd.notna(dict_morph) and ('-' in dict_morph or '+' in dict_morph):
            # 경계 유형 정보 포함
            morphemes_kor, boundaries = parse_dict_morph_with_boundaries(dict_morph)
            morphemes_roman = parse_word_roman_by_morphemes(word_roman, morphemes_kor)

            if len(morphemes_kor) >= 2 and len(morphemes_roman) >= 2:
                boundary_envs = check_nl_ln_at_boundary(morphemes_kor, morphemes_roman)

                for env in boundary_envs:
                    # 해당 위치의 경계 유형 가져오기
                    morph_boundary_type = boundaries[env['position']] if env['position'] < len(boundaries) else ''

                    # 어종 분류: classify_compound_type_from_row (word_type + seg_links 기반)
                    ct, morph1_origin, morph2_origin = classify_compound_type_from_row(row, sense_origin_dict)
                    compound_type = ct

                    env_type = env['env_type']
                    c1 = env['consonant1']
                    c2 = env['consonant2']

                    candidate = {
                        'word': word,
                        'word_stem': row['word_stem'],
                        'sense_id': row.get('sense_id', ''),
                        'boundary_type': 'morpheme',
                        'env_type': env_type,
                        'consonant1': c1,
                        'consonant2': c2,
                        'dict_morph': dict_morph,
                        'word_roman': word_roman,
                        'morph1': env['morph1'],
                        'morph2': env['morph2'],
                        'morph1_roman': env['morph1_roman'],
                        'morph2_roman': env['morph2_roman'],
                        'position': env['position'],

                        # 형태론 (어종 정보)
                        'morph1_origin': morph1_origin,
                        'morph2_origin': morph2_origin,
                        'compound_type': compound_type,

                        # 경계 유형 (합성/파생)
                        'morph_boundary_type': morph_boundary_type,

                        'seg_morph': row.get('seg_morph', ''),
                        'anal_morph': row.get('anal_morph', ''),
                        'pron': str(row.get('pron', '')) if pd.notna(row.get('pron', '')) else '',
                        'pron_roman': pron_roman,
                        'definition': row.get('definition', ''),
                        'freq_LS_total': row.get('freq_LS_total', 0),
                        'freq_MP_total': row.get('freq_MP_total', 0),
                        'freq_06b': row.get('freq_06b', 0),
                        'freq_13a': row.get('freq_13a', 0),
                    }

                    # 음운 과정 예상/감지
                    candidate['expected_process'] = get_expected_process(env_type)
                    candidate['detected_process'] = detect_assimilation_from_pron(c1, c2, env_type, word_roman, pron_roman)

                    candidates.append(candidate)

        # 2. 단어 내부 검색 (모든 명사)
        internal_envs = check_nl_ln_internal(word_roman)

        for env in internal_envs:
            env_type = env['env_type']
            c1 = env['consonant1']
            c2 = env['consonant2']

            candidate = {
                'word': word,
                'word_stem': row['word_stem'],
                'sense_id': row.get('sense_id', ''),
                'boundary_type': 'internal',
                'env_type': env_type,
                'consonant1': c1,
                'consonant2': c2,
                'dict_morph': dict_morph if pd.notna(dict_morph) else '',
                'word_roman': word_roman,
                'morph1': '',  # 내부는 형태소 정보 없음
                'morph2': '',
                'morph1_roman': env['syllable1'],
                'morph2_roman': env['syllable2'],
                'position': env['position'],

                # 어종 정보 (내부는 비워둠)
                'morph1_origin': '',
                'morph2_origin': '',
                'compound_type': '',

                # 경계 유형 (내부는 비워둠)
                'morph_boundary_type': '',

                'seg_morph': row.get('seg_morph', ''),
                'anal_morph': row.get('anal_morph', ''),
                'pron': str(row.get('pron', '')) if pd.notna(row.get('pron', '')) else '',
                'pron_roman': pron_roman,
                'definition': row.get('definition', ''),
                'freq_LS_total': row.get('freq_LS_total', 0),
                'freq_MP_total': row.get('freq_MP_total', 0),
                'freq_06b': row.get('freq_06b', 0),
                'freq_13a': row.get('freq_13a', 0),
            }

            # 음운 과정 예상/감지
            candidate['expected_process'] = get_expected_process(env_type)
            candidate['detected_process'] = detect_assimilation_from_pron(c1, c2, env_type, word_roman, pron_roman)

            candidates.append(candidate)

    return pd.DataFrame(candidates)

print("검색 함수 준비 완료 (sense_no 기반 어종 분류 + expected/detected_process 포함)")

검색 함수 준비 완료 (sense_no 기반 어종 분류 + expected/detected_process 포함)


In [12]:
# 5.2 검색 실행
print("ㄴㄹ/ㄹㄴ/비음ㄹ/저해음ㄹ/장애음ㄴ/장애음ㅁ 환경 검색 중...\n")
df_candidates = search_nl_ln_candidates(df_v7, sense_origin_dict)

print(f"\n검색 완료: {len(df_candidates):,}개 후보")
print(f"\n【환경 유형별】")
print(df_candidates['env_type'].value_counts())
print(f"\n【경계 유형별】")
print(df_candidates['boundary_type'].value_counts())

# 형태소 경계만 어종 통계
df_morpheme = df_candidates[df_candidates['boundary_type'] == 'morpheme']
if len(df_morpheme) > 0:
    print(f"\n【합성어 유형별 (형태소 경계만)】")
    print(df_morpheme['compound_type'].value_counts())

    print(f"\n【expected_process 분포】")
    print(df_candidates['expected_process'].value_counts())

    print(f"\n【detected_process 분포】")
    print(df_candidates['detected_process'].value_counts())

ㄴㄹ/ㄹㄴ/비음ㄹ/저해음ㄹ/장애음ㄴ/장애음ㅁ 환경 검색 중...

명사: 395,985개
word_roman 있는 명사: 395,974개

검색 완료: 29,218개 후보

【환경 유형별】
env_type
비음ㄹ     7788
장애음ㅁ    7036
ㄴㄹ      4844
저해음ㄹ    4374
장애음ㄴ    3304
ㄹㄴ      1872
Name: count, dtype: int64

【경계 유형별】
boundary_type
internal    21639
morpheme     7579
Name: count, dtype: int64

【합성어 유형별 (형태소 경계만)】
compound_type
S+S    4452
N+N    1857
U+U    1176
L+L      94
Name: count, dtype: int64

【expected_process 분포】
expected_process
lateralization            14504
obstruent_nasalization    10340
nasalization               4374
Name: count, dtype: int64

【detected_process 분포】
detected_process
unknown      20580
nasalized     8638
Name: count, dtype: int64


In [13]:
# 5.3 결과 미리보기
print("\n형태소 경계 상위 20개 (빈도순):\n")
df_boundary = df_candidates[df_candidates['boundary_type'] == 'morpheme']
df_boundary[['word', 'env_type', 'dict_morph', 'consonant1', 'consonant2', 'pron', 'freq_LS_total']].sort_values('freq_LS_total', ascending=False).head(20)


형태소 경계 상위 20개 (빈도순):



,word,env_type,dict_morph,consonant1,consonant2,pron,freq_LS_total
23850,옛날,장애음ㄴ,옛-날,t,n,옌ː날,534
8238,국물,장애음ㅁ,국-물,k,m,궁물,514
573,경쟁력,비음ㄹ,경쟁-력,ng,r,경ː쟁녁,411
23768,영향력,비음ㄹ,영향-력,ng,r,영ː향녁,207
16753,거짓말,장애음ㅁ,거짓-말,t,m,거ː진말,204
19887,성장률,비음ㄹ,성장-률,ng,r,성장뉼,184
20639,수익률,저해음ㄹ,수익-률,k,r,수잉뉼,169
21172,시청률,비음ㄹ,시청-률,ng,r,시ː청뉼,160
17005,보험료,비음ㄹ,보험-료,m,r,보ː험뇨,151
26209,입맛,장애음ㅁ,입-맛,p,m,임맏,142


---

## 6️⃣ 결과 저장

In [14]:
# 6.1 CSV 저장
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'{RESULT_DIR}/nl_ln_nasalization_candidates_{timestamp}.csv'

# 검토용 빈 컬럼 추가
df_candidates['review_status'] = 'pending'
df_candidates['notes'] = ''

# 컬럼 순서 정리
column_order = [
    'word', 'word_stem', 'sense_id',
    'boundary_type', 'env_type',
    'expected_process', 'detected_process',
    'consonant1', 'consonant2',
    'dict_morph', 'word_roman',
    'morph1', 'morph2', 'morph1_roman', 'morph2_roman', 'position',

    # 형태론 (어종 정보 - 형태소 경계만)
    'morph1_origin', 'morph2_origin', 'compound_type',

    # 경계 유형 (합성/파생 - 형태소 경계만)
    'morph_boundary_type',

    'seg_morph', 'anal_morph',
    'pron', 'pron_roman',
    'freq_LS_total', 'freq_MP_total', 'freq_06b', 'freq_13a',
    'definition',
    'review_status', 'notes'
]

df_full = df_candidates[column_order].sort_values(['boundary_type', 'freq_LS_total'], ascending=[True, False])
df_full.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"결과 저장: {output_path}")
print(f"   총 {len(df_full):,}개 후보")
print(f"\n개선 사항:")
print(f"   - utils_phonology.py 공통 모듈 사용")
print(f"   - sense_no 기반 어종 분류 (classify_compound_type_from_row)")
print(f"   - expected_process: 이론적 예상 음운 변화")
print(f"   - detected_process: pron_roman 기반 실제 감지")
print(f"   - 장애음ㄴ, 장애음ㅁ 환경 추가")

결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/nl_ln_nasalization_candidates_20260312_070559.csv
   총 29,218개 후보

개선 사항:
   - utils_phonology.py 공통 모듈 사용
   - sense_no 기반 어종 분류 (classify_compound_type_from_row)
   - expected_process: 이론적 예상 음운 변화
   - detected_process: pron_roman 기반 실제 감지
   - 장애음ㄴ, 장애음ㅁ 환경 추가


In [15]:
# 6.2 통계 요약
print("\n" + "="*70)
print("📊 유음화/비음화 환경 검색 요약")
print("="*70)

print(f"\n총 후보 수: {len(df_full):,}개")

print(f"\n【환경 유형별】")
print(df_full['env_type'].value_counts())

print(f"\n【경계 유형별】")
print(df_full['boundary_type'].value_counts())

print(f"\n【환경×경계 교차표】")
print(pd.crosstab(df_full['env_type'], df_full['boundary_type']))

print("\n="*70)


📊 유음화/비음화 환경 검색 요약

총 후보 수: 29,218개

【환경 유형별】
env_type
비음ㄹ     7788
장애음ㅁ    7036
ㄴㄹ      4844
저해음ㄹ    4374
장애음ㄴ    3304
ㄹㄴ      1872
Name: count, dtype: int64

【경계 유형별】
boundary_type
internal    21639
morpheme     7579
Name: count, dtype: int64

【환경×경계 교차표】
boundary_type  internal  morpheme
env_type                         
ㄴㄹ                 4024       820
ㄹㄴ                 1213       659
비음ㄹ                6124      1664
장애음ㄴ               2124      1180
장애음ㅁ               4829      2207
저해음ㄹ               3325      1049

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
